In [10]:
import pandas as pd
import networkx as nx

from utils.geometry_utils import rebuild_polygons
from utils.graph_utils import build_graph

In [11]:
rooms_df = pd.read_csv("rooms_dataset.csv")
rooms_df = rebuild_polygons(rooms_df)

In [1]:
public_rooms = ["livingroom","diningroom","entry","entry lobby","terrace"]

private_rooms = ["bedroom","master bedroom"]

service_rooms = ["kitchen","utility","laundry","pantry","bath","bathroom","toilet"]

In [13]:
def public_private_separation(G):
    if G.number_of_nodes() == 0:
        return 0
    violations = 0
    total_edges = G.number_of_edges()
    if total_edges == 0:
        return 0
    for u, v in G.edges():
        r1 = G.nodes[u]["room_type"]
        r2 = G.nodes[v]["room_type"]
        if (r1 in public_rooms and r2 in private_rooms) or (r2 in public_rooms and r1 in private_rooms):
            violations += 1
    return 1 - (violations / total_edges)  # higher is better separation

In [14]:
def service_area_ratio(G):
    service_count = sum(
    1 for n, d in G.nodes(data=True)
    if d["room_type"].lower() in service_rooms
)
    total_count = G.number_of_nodes()
    if total_count == 0:
        return 0
    return service_count / total_count

In [15]:
def bathroom_adjacency(G):
    bedrooms = [
        n for n, d in G.nodes(data=True)
        if "bedroom" in d["room_type"].lower()
    ]

    bathrooms = [
        n for n, d in G.nodes(data=True)
        if "bath" in d["room_type"].lower()
    ]
    if len(bedrooms) == 0 or len(bathrooms) == 0:
        return 0
    adj_count = 0
    for bed in bedrooms:
        for bath in bathrooms:
            if G.has_edge(bed, bath):
                adj_count += 1
    return adj_count / len(bedrooms)

In [16]:
results = []

for plan_id in rooms_df["plan_id"].unique():
    plan_rooms = rooms_df[rooms_df["plan_id"] == plan_id]
    G = build_graph(plan_rooms)
    
    metrics = {}
    metrics["plan_id"] = plan_id
    metrics["public_private_separation"] = public_private_separation(G)
    metrics["service_area_ratio"] = service_area_ratio(G)
    metrics["bathroom_adjacency"] = bathroom_adjacency(G)
    
    results.append(metrics)

In [17]:
functional_features_df = pd.DataFrame(results)
functional_features_df.head()

,plan_id,public_private_separation,service_area_ratio,bathroom_adjacency
0,6044,1.0,0.300000,0.0
1,2564,1.0,0.000000,0.0
2,6165,1.0,0.166667,0.0
3,1021,1.0,0.058824,1.0
4,5507,1.0,0.125000,0.0


In [18]:
functional_features_df.to_csv("functional_zoning_features.csv", index=False)